# 006 — Actions complètes, séquences et optimisation du F1

Objectif : mesurer si les liens entre actions et contextes, leur ordre et le rythme améliorent les spécialistes Extra Trees de `005`. La cible de 97 % est un objectif, pas un résultat garanti.

Le code réutilisable est dans `sequence_experiment.py`. Ce notebook peut être exécuté depuis le dossier du projet ou depuis `datamanip`. Les résultats sont écrits dans `datamanip/resultats_006`.


## Protocole fixé avant les résultats

- Même séparation stratifiée 80/20, graine 42, que `003`–`005`.
- Sélection sur trois plis de la partie apprentissage, graine 42, selon le F1 macro moyen des plis.
- Encodage, vocabulaire, filtrage des termes rares et IDF ajustés sur chaque pli d’apprentissage uniquement.
- Extra Trees : référence et caractéristiques enrichies, 200 arbres, `max_features=0.5`, classes équilibrées, spécialistes par navigateur.
- SVM : actions complètes seules ou avec séquences ; `C` dans 1, 5, 20 ; fréquence sous-linéaire et normalisation L2.
- Ensemble : meilleur arbre et meilleur SVM selon la CV ; poids SVM dans 0.25, 0.5, 0.75. Softmax de trois fois la marge SVM : score de combinaison, pas probabilité calibrée.
- La configuration choisie reste fixe pour le contrôle de sensibilité avec la graine 137, puis pour la validation réservée.

La seconde graine réutilise les données de sélection : ce contrôle ne remplace pas une validation imbriquée. Les 20 % réservés ont déjà été examinés dans les notebooks précédents : leur score n’est pas une estimation finale indépendante. Le test Kaggle n’intervient pas. Le type de moyenne F1 officiel reste à confirmer.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data/train.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "datamanip/sequence_experiment.py").is_file()
sys.path.insert(0, str(ROOT / "datamanip"))
from sequence_experiment import prepare, search, validate, OUT
import pandas as pd


## Construire les représentations

Les anciennes caractéristiques sont reconstruites depuis `002_features.ipynb`. Les caractéristiques supplémentaires sont calculées session par session, sans utiliser l’identité :

| Représentation | Contenu |
|---|---|
| `base` | Caractéristiques exactes de `002` |
| `full` | Comptages séparés et actions complètes avec leurs contextes |
| `seq` | `full`, paires et triplets d’actions complètes et de types d’actions, transitions entre mentions d’écrans |
| `enriched` | `base`, actions complètes, paires et triplets de types, transitions et statistiques du nombre d’actions entre marqueurs |

Les séquences conservent l’ordre des actions après retrait des marqueurs temporels. Les transitions concernent les mentions d’écrans et ne prétendent pas reconstituer l’écran actif. Le rythme décrit des comptages entre marqueurs, sans supposer que `tXX` représente une durée élémentaire.


In [ ]:
data = prepare()


## Recherche sur les plis d’apprentissage

Les scores sont enregistrés progressivement. Le F1 macro moyen des plis sert à sélectionner ; le F1 calculé sur les prédictions hors pli regroupées est également conservé mais peut différer.


In [ ]:
experiment = search(data)
experiment["table"]


## Sensibilité et validation de la configuration retenue

Le choix n’est pas changé après observation de la seconde graine ou des 20 % de validation. Les corrections et régressions par rapport à la référence sont exportées, ainsi que les scores par utilisateur.


In [ ]:
resultats, diagnostic = validate(data, experiment)
resultats


## Examiner les erreurs et les classes rares


In [ ]:
diagnostic.loc[diagnostic.erreur].sort_values(
    ["sessions_apprentissage", "nb_actions"]
).head(25)


In [ ]:
pd.read_csv(OUT / "rapport_utilisateurs.csv", index_col=0).sort_values("f1-score").head(20)


## Relire les résultats sans relancer les modèles

Les fichiers CSV et `resume.json` peuvent être consultés directement. Pour une nouvelle recherche, exécuter le notebook dans l’ordre. Les scores enregistrés par l’exécution terminale sont résumés dans `resultats_006/COMPTE_RENDU.md`.


## Résultats mesurés — exécution terminale du 18 octobre 2026

| Modèle | F1 macro (%) | F1 pondéré (%) | Accuracy (%) |
| --- | --- | --- | --- |
| Extra Trees reference | 89.05 | 90.42 | 91.46 |
| Ensemble poids SVM=0.75 | 91.87 | 93.41 | 94.21 |

**F1 macro validation : 91,87 %, soit +2,82 points.** L’ensemble retenu combine 75 % de score SVM séquentiel (`C=20`) et 25 % d’Extra Trees de référence. F1 macro CV : **92,41 %** ; sur le second découpage : **91,65 %**, contre **86,76 %** pour la référence. Les 18 corrections n’introduisent aucune régression sur la validation ; il reste 38 erreurs. **L’objectif de 97 % n’est pas atteint.**

Le compte rendu complet et les diagnostics sont dans `resultats_006`. Ces résultats proviennent du script exécuté au terminal ; les cellules ci-dessus sont prêtes à être réexécutées. Le second découpage n’est pas une évaluation indépendante et les 20 % réservés ont déjà été examinés lors des expériences précédentes.
